In [1]:
import pg8000
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, MetaData, Table, select, insert
from sqlalchemy.exc import SQLAlchemyError
load_dotenv()
import pandas as pd

In [2]:
DB_HOST = os.getenv("DB_HOST")
NC_ACC = os.getenv("NC_ACC")
NC_PASS = os.getenv("NC_PASS")

def create_db_connection():
    DB_HOST = os.getenv("DB_HOST")
    DB_NAME = os.getenv("DB_NAME")
    DB_USER = os.getenv("DB_USER")
    DB_PASSWORD = os.getenv("DB_PASSWORD")
    engine = create_engine('postgresql+pg8000://'+DB_USER+':'+DB_PASSWORD+'@'+DB_HOST+':5432/'+DB_NAME)
    return engine

In [22]:
engine = create_db_connection()
id= 2158
       
df_hypo = pd.read_sql(f'''SELECT *
                            FROM bre_advance_search
                            WHERE bre_advance_search.id = '{id}' ''', engine)
df_tags = pd.read_sql(f'''SELECT oc_systemtag.name AS tag_name
                                    FROM oc_systemtag_object_mapping
                                    JOIN oc_systemtag ON oc_systemtag_object_mapping.systemtagid = oc_systemtag.id
                                    WHERE oc_systemtag_object_mapping.objectid = '{id}' ''', engine)

In [23]:
df_tags

,tag_name
0,flower
1,nature
2,still
3,life
4,vase
5,red
6,white
7,orange
8,green
9,bloom


In [24]:
df_hypo

,index,id,is_color,is_adjective,lvl3_hyponym,hyponym_all


In [ ]:
    for idx, col in enumerate(cols):
        with col:
            len_col = len(df_data) // N_of_cols
            start_idx = idx * len_col
            end_idx = (idx + 1) * len_col if idx != N_of_cols - 1 else len(df_data)
            display_data = df_data[start_idx:end_idx]
            for index, row in display_data.iterrows():
                try:
                    file_id = row['fileid']
                    img_path = row['preview_url']
                    tag_names = row.get('tagnames', '')

                    st.image(get_images(file_id, img_path))
                    st.write(f"{tag_names}")
                except Exception as e:
                    print(f"Error loading image for file_id {file_id}: {e}")
       #try:

           # st.write(f"hypo: ")
           # st.dataframe(get_hypo_search(file_id))
        #except Exception as e:
         #   print(f"no Preview availible for:{img_path} /n: {e}")


In [ ]:
def get_images(file_id,file_path):
    DB_HOST = os.getenv("DB_HOST")
    NC_ACC = os.getenv("NC_ACC")
    NC_PASS = os.getenv("NC_PASS")

    server_url = f'''http://{DB_HOST}:8080/remote.php/dav/files/{NC_ACC}'''
    preview_url = f'''http://{DB_HOST}:8080/core/preview?fileId={file_id}&x=1080&y=1080'''
    username = NC_ACC
    password = NC_PASS

    # Send a GET request to download the file
    response = requests.get(preview_url, auth=HTTPBasicAuth(username, password), stream=True)
 
    # Check if the request was successful
    if response.status_code == 200:
        # Save the file content in memory using BytesIO
        file_in_memory = BytesIO()
        for chunk in response.iter_content(chunk_size=1024):
            if chunk:
                file_in_memory.write(chunk)
        # Open the image using Pillow (PIL)
        img = PILImage.open(file_in_memory)
        return img ,file_id
    elif response.status_code == 404:
        print(f"no preview availible for file: {file_id} {file_path}")
    else:
        print(f"Failed to download file. Status code: {response.status_code}")
        print(response.text)
    try:
        import gc
        del file_in_memory
        gc.collect()
    except:
        pass

In [1]:
import regex as re
from nltk.corpus import stopwords
import spacy
from time import sleep


try:
    nlp = spacy.load("en_core_web_trf")
except:
    print('failed to load spacy lemmatizer')
    print('stopping execution')
    sleep(10)
try:
    stop_words = set(stopwords.words('english'))
    stop_words.update(['art','color','colorcomposition'])
except:
    print('failed to load nltk stopwords')
    print('stopping execution')
    sleep(10)

def modulate_search_phrase(input_str):
    input_str = input_str.str.split()
     




/Users/tom/Fine-arts-ML/Fine-Arts-Main/.venv3_12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
input_str = 'green and small flower-gardens'

input_str = re.sub('[^A-Za-z0-9]+', ' ', input_str)
input_list = input_str.split()
input_lemma_list = []
for word in input_list:
    doc = nlp(word)
    input_lemma_list.append(" ".join([token.lemma_ for token in doc]))
input_lemma_no_stop = [word for word in input_lemma_list if word not in stop_words]
output_search_str = '|'.join(input_lemma_no_stop)

In [18]:
engine = create_db_connection()
df_color_ids = pd.read_sql(f'''
    SELECT is_color, id
    FROM bre_advance_search
    WHERE bre_advance_search.is_color = TRUE
''', engine)
df_color_names = pd.read_sql(f'''
    SELECT id, name
    FROM oc_systemtag
''',engine)

In [28]:
df_colors = df_color_names.merge(df_color_ids, on = 'id', how= 'left')

In [ ]:
df_colors = df_colors[df_colors['is_color'] == True].drop(columns='is_color')

In [ ]:
df_colors


,id,name
5,824,red
6,825,white
7,826,orange
8,827,green
19,839,pink
31,851,black
40,860,blush
45,865,blue
46,866,yellow
57,877,gray


In [7]:
engine = create_db_connection()
metadata = MetaData()
bre_advance_search = Table('bre_advance_search', metadata, autoload_with=engine)

query = select(
    bre_advance_search.c.id,
    bre_advance_search.c.is_color
).where(
    bre_advance_search.c.is_color == True
)

with engine.connect() as connection:
    result = connection.execute(query)
    color_ids = [{'id':row.id,'is_color': row.is_color} for row in result]


# Reflect the table
oc_systemtag = Table('oc_systemtag', metadata, autoload_with=engine)

# Build the query
query = select(
    oc_systemtag.c.id,
    oc_systemtag.c.name
)

# Execute the query and fetch results
with engine.connect() as connection:
    result = connection.execute(query)
    color_names = [{'id': row.id, 'name': row.name} for row in result]
df_color_names = pd.DataFrame(color_names)
df_color_ids = pd.DataFrame(color_ids)

df_colors = df_color_names.merge(df_color_ids, on = 'id', how= 'left')
df_colors = df_colors[df_colors['is_color'] == True].drop(columns='is_color')
df_colors.rename(columns={'id':'id','name':'Color'})

,id,Color
5,824,red
6,825,white
7,826,orange
8,827,green
19,839,pink
31,851,black
40,860,blush
45,865,blue
46,866,yellow
57,877,gray
